# NexTwin AI — Industrial Digital Twin Platform
## Notebook 04: Machine Health & Failure Prediction Model

### Objectives
1. **Load Engineered Dataset**: Load `engineered_machine_health.csv` (AI4I dataset).
2. **Train Binary Classifiers**: Predict `machine_failure` using:
   - Random Forest Classifier
   - XGBoost Classifier
   - LightGBM Classifier
3. **Address Class Imbalance**: Incorporate class weights or scale position weights.
4. **Hyperparameter Tuning**: Perform cross-validated search (`GridSearchCV`) for optimal model parameters.
5. **Model Evaluation**: Compare models using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
6. **Export Model**: Serialize the best performing classifier as `health_model.pkl`.

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Dataset & Train-Test Split
We load the engineered AI4I machine health dataset and split it into training and testing partitions, ensuring stratification on the target label.

In [2]:
PROCESSED_DIR = os.path.join("..", "datasets", "processed")
df = pd.read_csv(os.path.join(PROCESSED_DIR, "engineered_machine_health.csv"))

# Select features and target
feature_cols = [
    'type', 'air_temperature', 'process_temperature', 'rotational_speed', 
    'torque', 'tool_wear', 'machine_health_score', 'failure_risk_index'
]
target_col = 'machine_failure'

X = df[feature_cols]
y = df[target_col]

# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Failure rate in training: {y_train.mean()*100:.2f}%")
print(f"Failure rate in test: {y_test.mean()*100:.2f}%")

Training set shape: (8000, 8)
Test set shape: (2000, 8)
Failure rate in training: 3.39%
Failure rate in test: 3.40%


## 2. Define Preprocessor Pipeline
We build a column transformer to scale numerical variables and encode the categorical machine `type` variable.

In [3]:
num_cols = ['air_temperature', 'process_temperature', 'rotational_speed', 'torque', 'tool_wear', 'machine_health_score', 'failure_risk_index']
cat_cols = ['type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

print("Preprocessor pipeline defined.")

Preprocessor pipeline defined.


## 3. Train Base Classifiers
We train Random Forest, XGBoost, and LightGBM models. To account for the class imbalance, we adjust class/position weights in all architectures.

In [4]:
# Compute scale position weight for XGBoost/LightGBM
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

models = {
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)
}

results = {}

for name, model in models.items():
    # Create complete training pipeline
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Fit the pipeline
    clf.fit(X_train, y_train)
    
    # Predictions
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    # Metrics
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Pipeline": clf
    }
    
    print(f"=== {name} Evaluated ===")
    print(classification_report(y_test, y_pred))

=== Random Forest Evaluated ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.82      0.40      0.53        68

    accuracy                           0.98      2000
   macro avg       0.90      0.70      0.76      2000
weighted avg       0.97      0.98      0.97      2000



=== XGBoost Evaluated ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.64      0.68      0.66        68

    accuracy                           0.98      2000
   macro avg       0.81      0.83      0.82      2000
weighted avg       0.98      0.98      0.98      2000



=== LightGBM Evaluated ===
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1932
           1       0.60      0.76      0.67        68

    accuracy                           0.97      2000
   macro avg       0.79      0.87      0.83      2000
weighted avg       0.98      0.97      0.98      2000



C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 4. Hyperparameter Tuning (XGBoost)
Let's perform Grid Search optimization on the XGBoost classifier to fine-tune its performance.

In [5]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'))
])

param_grid = {
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__n_estimators': [50, 100, 150]
}

grid_search = GridSearchCV(
    xgb_pipeline, param_grid, cv=3, scoring='f1', verbose=1, n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)
y_proba_tuned = tuned_model.predict_proba(X_test)[:, 1]

results["Tuned XGBoost"] = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(y_test, y_pred_tuned),
    "Recall": recall_score(y_test, y_pred_tuned),
    "F1-Score": f1_score(y_test, y_pred_tuned),
    "ROC-AUC": roc_auc_score(y_test, y_proba_tuned),
    "Pipeline": tuned_model
}

Fitting 3 folds for each of 27 candidates, totalling 81 fits


Best parameters found: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 7, 'classifier__n_estimators': 100}
Best CV F1-Score: 0.6274


## 5. Model Comparison & Leaderboard
We summarize performance across all models in a clean tabular layout.

In [6]:
metrics_df = pd.DataFrame(results).T.drop(columns=['Pipeline'])
print("=== Machine Failure Model Leaderboard ===")
display(metrics_df.sort_values(by="F1-Score", ascending=False))

=== Machine Failure Model Leaderboard ===


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Tuned XGBoost,0.977,0.6375,0.75,0.689189,0.974721
LightGBM,0.9745,0.597701,0.764706,0.670968,0.971311
XGBoost,0.976,0.638889,0.676471,0.657143,0.973709
Random Forest,0.9765,0.818182,0.397059,0.534653,0.943327


## 6. Exporting Best Model
We pick the best model from the leaderboard and save it as a serialized pickle file for inference.

In [7]:
# Identify best model based on F1-Score
best_model_name = metrics_df['F1-Score'].idxmax()
best_pipeline = results[best_model_name]['Pipeline']

print(f"Best Model Selected: {best_model_name}")

# Export pipeline
model_path = "health_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(best_pipeline, f)

print(f"Serialized model successfully exported to: {os.path.abspath(model_path)}")

Best Model Selected: Tuned XGBoost
Serialized model successfully exported to: D:\PROJECTS\NexTwinAI\notebooks\health_model.pkl
